In [1]:
import pandas as pd
import tqdm, os
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../../..")

from utils.information_theoretic_calculation import InformationTheoreticCalculator
from utils.data_processing import format_context_with_boundaries
from utils.ipa_processing import syllabify_ipa
SEP = "_"

In [2]:
df = pd.read_csv("../dataset_construction/datasets/processed_frmri.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])
df = format_context_with_boundaries(df, lang="fr")

In [3]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,2.647,2.717,2.717,2.907,f,i,f,filou,VIDEO_P6_S1.avi_2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,S1.avi
1,2.717,2.907,2.907,2.937,i,l,f_i_.,filou,VIDEO_P6_S1.avi_2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,S1.avi
2,2.907,2.937,2.937,3.067,l,u,f_i_._l,filou,VIDEO_P6_S1.avi_2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,S1.avi
3,3.097,3.157,3.157,3.237,l,a,l,la,VIDEO_P6_S1.avi_4,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,S1.avi
4,3.237,3.317,3.317,3.377,f,ʁ,f,fripouille,VIDEO_P6_S1.avi_5,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,S1.avi
...,...,...,...,...,...,...,...,...,...,...,...,...
22432,33.071,33.131,33.131,33.211,ʁ,e,p_ʁ,précises,VIDEO_P4_S12.avi_78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,S12.avi
22433,33.131,33.211,33.211,33.391,e,s,p_ʁ_e_.,précises,VIDEO_P4_S12.avi_78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,S12.avi
22434,33.211,33.391,33.391,33.671,s,i,p_ʁ_e_._s,précises,VIDEO_P4_S12.avi_78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,S12.avi
22435,33.391,33.671,33.671,33.801,i,z,p_ʁ_e_._s_i_.,précises,VIDEO_P4_S12.avi_78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,S12.avi


In [4]:
with open("../../pronunciation_counts/pronunciation_counts_fr.pkl", "rb") as f:
    pronunciation_counts = pickle.load(f)
pronunciation_counts = {syllabify_ipa(k, lang="fr").replace(".", "_._"): v for k, v in pronunciation_counts.items() if "*" not in k and k != ""}

In [5]:
bigrams = set()
for phone1, phone2 in tqdm.tqdm(zip(df["phone1"], df["phone2"]), total=len(df)):
    bigrams.add(phone1+SEP+phone2)

all_phones = set()
for pronunciation in tqdm.tqdm(pronunciation_counts):

    for phone in pronunciation.split(SEP):
        all_phones.add(phone)

monogram_counts = {}
for pronunciation, count in tqdm.tqdm(pronunciation_counts.items()):
    phones = pronunciation.split(SEP)
    for phone in phones:
        if phone not in monogram_counts:
            monogram_counts[phone] = 0
        monogram_counts[phone] += count

100%|██████████| 7193272/7193272 [00:08<00:00, 863174.97it/s]


In [6]:
calculator = InformationTheoreticCalculator(
    pronunciation_counts=pronunciation_counts,
    phone_counts=monogram_counts,
    sep=SEP
)

100%|██████████| 7193272/7193272 [00:56<00:00, 127076.88it/s]


In [7]:
data = {
    "bigram_surprisal": [],
    "phone2_pos_entropy": [],
}

In [10]:
data["bigram_surprisal"] = df.apply(lambda x: calculator.get_contextual_surprisal(SEP.join(x["context"].split(SEP)[:-1]), x["phone1"] + SEP + x["phone2"]), axis=1).tolist()
data["phone2_pos_entropy"] = df.apply(lambda x: calculator.get_target_pos_entropy(x["context"]), axis=1).tolist()
data["phone2_freq"] = df.apply(lambda x: monogram_counts.get(x["phone2"], None), axis=1).tolist()

In [11]:
pd.DataFrame(data).to_csv("surprisal_frmri.csv", index=False)